# End-to-End CLI Workflow

This notebook exercises the repository through the documented command-line entrypoints:

- synthetic demand generation
- baseline experiment run
- congestion-aware experiment run
- congestion-focused benchmark run

It writes all outputs under `outputs/notebooks/e2e_cli/`.

In [1]:
from __future__ import annotations

import csv
import json
import os
import subprocess
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists():
    REPO_ROOT = REPO_ROOT.parent
assert (REPO_ROOT / "src").exists(), "Run this notebook from the repository root or notebooks/ directory."

OUTPUT_ROOT = REPO_ROOT / "outputs" / "notebooks" / "e2e_cli"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(REPO_ROOT)

def run_command(args: list[str]) -> subprocess.CompletedProcess[str]:
    env = dict(os.environ)
    existing = env.get("PYTHONPATH", "")
    env["PYTHONPATH"] = str(REPO_ROOT / "src") if not existing else f"{REPO_ROOT / 'src'}{os.pathsep}{existing}"
    completed = subprocess.run(
        args,
        cwd=REPO_ROOT,
        env=env,
        text=True,
        capture_output=True,
        check=False,
    )
    print("$", " ".join(args))
    if completed.stdout:
        print(completed.stdout)
    if completed.returncode != 0:
        if completed.stderr:
            print(completed.stderr)
        raise RuntimeError(f"Command failed with exit code {completed.returncode}: {' '.join(args)}")
    return completed

print(REPO_ROOT)
print(OUTPUT_ROOT)


/Users/adityadutta/Desktop/GitHub/GNN-Warehouse-Sim
/Users/adityadutta/Desktop/GitHub/GNN-Warehouse-Sim/outputs/notebooks/e2e_cli


In [3]:
demand_output = OUTPUT_ROOT / "task_demand.csv"

run_command([
    "python3",
    "-m",
    "warehouse_sim.demand.cli",
    "--output",
    str(demand_output),
    "--horizon-seconds",
    "1800",
    "--mean-interval",
    "90",
    "--min-tasks",
    "8",
    "--rush-start",
    "1800",
    "--rush-end",
    "1800",
    "--lunch-start",
    "1800",
    "--lunch-end",
    "1800",
    "--include-task-metadata",
])

with demand_output.open() as handle:
    reader = csv.DictReader(handle)
    rows = list(reader)

print(f"Generated rows: {len(rows)}")
print(rows[:2])


$ python3 -m warehouse_sim.demand.cli --output /Users/adityadutta/Desktop/GitHub/GNN-Warehouse-Sim/outputs/notebooks/e2e_cli/task_demand.csv --horizon-seconds 1800 --mean-interval 90 --min-tasks 8 --rush-start 1800 --rush-end 1800 --lunch-start 1800 --lunch-end 1800 --include-task-metadata
Wrote 21 tasks to /Users/adityadutta/Desktop/GitHub/GNN-Warehouse-Sim/outputs/notebooks/e2e_cli/task_demand.csv
Shift horizon: 1800 sec
Observed mean interarrival: 80.956 sec
95th percentile interarrival: 235.194 sec

Generated rows: 21
[{'Task_ID': '1', 'Timestamp': '57.424', 'Interarrival_Time': '57.424', 'Regime': 'base', 'Task_Type': 'pick', 'Source_Zone': 'storage_b', 'Destination_Zone': 'staging', 'Priority': '2', 'Service_Duration': '65.923'}, {'Task_ID': '2', 'Timestamp': '93.046', 'Interarrival_Time': '35.623', 'Regime': 'base', 'Task_Type': 'cycle_count', 'Source_Zone': 'storage_a', 'Destination_Zone': 'staging', 'Priority': '3', 'Service_Duration': '100.16'}]


In [4]:
baseline_output = OUTPUT_ROOT / "baseline_experiment"

run_command([
    "python3",
    "-m",
    "warehouse_sim.simulation.experiment_cli",
    "--config",
    "configs/baseline_experiment.toml",
    "--output-dir",
    str(baseline_output),
    "--write-observation-dataset",
])

baseline_summary = json.loads((baseline_output / "summary.json").read_text())
baseline_summary["metrics"]


$ python3 -m warehouse_sim.simulation.experiment_cli --config configs/baseline_experiment.toml --output-dir /Users/adityadutta/Desktop/GitHub/GNN-Warehouse-Sim/outputs/notebooks/e2e_cli/baseline_experiment --write-observation-dataset
Experiment: configs/baseline_experiment.toml
Policy: fifo
Tasks completed: 7
Output directory: /Users/adityadutta/Desktop/GitHub/GNN-Warehouse-Sim/outputs/notebooks/e2e_cli/baseline_experiment
summary: /Users/adityadutta/Desktop/GitHub/GNN-Warehouse-Sim/outputs/notebooks/e2e_cli/baseline_experiment/summary.json
executions: /Users/adityadutta/Desktop/GitHub/GNN-Warehouse-Sim/outputs/notebooks/e2e_cli/baseline_experiment/executions.csv
queue_snapshots: /Users/adityadutta/Desktop/GitHub/GNN-Warehouse-Sim/outputs/notebooks/e2e_cli/baseline_experiment/queue_snapshots.csv
robot_metrics: /Users/adityadutta/Desktop/GitHub/GNN-Warehouse-Sim/outputs/notebooks/e2e_cli/baseline_experiment/robot_metrics.csv
dataset_manifest: /Users/adityadutta/Desktop/GitHub/GNN-Wareho

{'tasks_generated': 7,
 'tasks_completed': 7,
 'tasks_unassigned': 0,
 'average_waiting_time': 6.006519188595608,
 'average_turnaround_time': 73.43509061716703,
 'average_travel_distance_per_task': 7.428571428571429,
 'realized_travel_time_total': 52.0,
 'realized_travel_distance_total': 52.0,
 'congestion_delay_total': 0.0,
 'average_congestion_delay_per_completed_task': 0.0,
 'blocked_traversal_events_total': 0,
 'average_queue_length': 0.08597720055858567,
 'throughput_per_hour': 51.53033101077584,
 'makespan': 489.0323719195645,
 'robot_metrics': [{'robot_id': 'robot_1',
   'tasks_completed': 5,
   'utilization': 0.6911607889540569,
   'busy_time': 338.0,
   'idle_time': 151.03237191956453,
   'travel_time': 38.0,
   'travel_distance': 38.0,
   'congestion_delay_time': 0.0,
   'blocked_traversal_events': 0},
  {'robot_id': 'robot_2',
   'tasks_completed': 2,
   'utilization': 0.2740104902953954,
   'busy_time': 134.00000000000003,
   'idle_time': 355.0323719195645,
   'travel_time'

In [5]:
congestion_output = OUTPUT_ROOT / "narrow_bottleneck"

run_command([
    "python3",
    "-m",
    "warehouse_sim.simulation.experiment_cli",
    "--config",
    "configs/scenarios/narrow_bottleneck.toml",
    "--output-dir",
    str(congestion_output),
])

congestion_summary = json.loads((congestion_output / "summary.json").read_text())
congestion_summary["metrics"]


$ python3 -m warehouse_sim.simulation.experiment_cli --config configs/scenarios/narrow_bottleneck.toml --output-dir /Users/adityadutta/Desktop/GitHub/GNN-Warehouse-Sim/outputs/notebooks/e2e_cli/narrow_bottleneck
Experiment: configs/scenarios/narrow_bottleneck.toml
Policy: fifo
Tasks completed: 11
Output directory: /Users/adityadutta/Desktop/GitHub/GNN-Warehouse-Sim/outputs/notebooks/e2e_cli/narrow_bottleneck
summary: /Users/adityadutta/Desktop/GitHub/GNN-Warehouse-Sim/outputs/notebooks/e2e_cli/narrow_bottleneck/summary.json
executions: /Users/adityadutta/Desktop/GitHub/GNN-Warehouse-Sim/outputs/notebooks/e2e_cli/narrow_bottleneck/executions.csv
queue_snapshots: /Users/adityadutta/Desktop/GitHub/GNN-Warehouse-Sim/outputs/notebooks/e2e_cli/narrow_bottleneck/queue_snapshots.csv
robot_metrics: /Users/adityadutta/Desktop/GitHub/GNN-Warehouse-Sim/outputs/notebooks/e2e_cli/narrow_bottleneck/robot_metrics.csv



{'tasks_generated': 11,
 'tasks_completed': 11,
 'tasks_unassigned': 0,
 'average_waiting_time': 0.0,
 'average_turnaround_time': 34.90909090909091,
 'average_travel_distance_per_task': 14.909090909090908,
 'realized_travel_time_total': 164.0,
 'realized_travel_distance_total': 164.0,
 'congestion_delay_total': 0.0,
 'average_congestion_delay_per_completed_task': 0.0,
 'blocked_traversal_events_total': 0,
 'average_queue_length': 0.0,
 'throughput_per_hour': 62.082156537347245,
 'makespan': 637.8644397795285,
 'robot_metrics': [{'robot_id': 'robot_1',
   'tasks_completed': 6,
   'utilization': 0.3323590198464045,
   'busy_time': 212.0,
   'idle_time': 425.86443977952854,
   'travel_time': 92.0,
   'travel_distance': 92.0,
   'congestion_delay_time': 0.0,
   'blocked_traversal_events': 0},
  {'robot_id': 'robot_2',
   'tasks_completed': 4,
   'utilization': 0.2194823715966822,
   'busy_time': 140.0,
   'idle_time': 497.86443977952854,
   'travel_time': 60.0,
   'travel_distance': 60.0,


In [6]:
benchmark_output = OUTPUT_ROOT / "congestion_benchmark"

run_command([
    "python3",
    "-m",
    "warehouse_sim.simulation.benchmark_cli",
    "--config",
    "configs/congestion_policy_benchmark.toml",
    "--output-dir",
    str(benchmark_output),
])

benchmark_summary = json.loads((benchmark_output / "benchmark_summary.json").read_text())
print("runs", len(benchmark_summary["runs"]))
benchmark_summary["best_by_scenario"]


$ python3 -m warehouse_sim.simulation.benchmark_cli --config configs/congestion_policy_benchmark.toml --output-dir /Users/adityadutta/Desktop/GitHub/GNN-Warehouse-Sim/outputs/notebooks/e2e_cli/congestion_benchmark
Benchmark: configs/congestion_policy_benchmark.toml
summary_csv: /Users/adityadutta/Desktop/GitHub/GNN-Warehouse-Sim/outputs/notebooks/e2e_cli/congestion_benchmark/benchmark_summary.csv
summary_json: /Users/adityadutta/Desktop/GitHub/GNN-Warehouse-Sim/outputs/notebooks/e2e_cli/congestion_benchmark/benchmark_summary.json

runs 12


{'narrow_bottleneck': {'scenario_name': 'narrow_bottleneck',
  'scenario_config': '/Users/adityadutta/Desktop/GitHub/GNN-Warehouse-Sim/configs/scenarios/narrow_bottleneck.toml',
  'seed': 23,
  'policy': 'fifo',
  'execution_model': 'reserved_edges',
  'tasks_generated': 11,
  'tasks_completed': 11,
  'tasks_unassigned': 0,
  'average_waiting_time': 0.0,
  'average_turnaround_time': 34.90909090909091,
  'average_travel_distance_per_task': 14.909090909090908,
  'realized_travel_time_total': 164.0,
  'realized_travel_distance_total': 164.0,
  'congestion_delay_total': 0.0,
  'average_congestion_delay_per_completed_task': 0.0,
  'blocked_traversal_events_total': 0,
  'average_queue_length': 0.0,
  'throughput_per_hour': 62.082156537347245,
  'makespan': 637.8644397795285,
  'summary_path': '/Users/adityadutta/Desktop/GitHub/GNN-Warehouse-Sim/outputs/notebooks/e2e_cli/congestion_benchmark/narrow_bottleneck/seed_23/fifo/summary.json'},
 'high_fleet_density': {'scenario_name': 'high_fleet_de

In [7]:
with (benchmark_output / "benchmark_summary.csv").open() as handle:
    rows = list(csv.DictReader(handle))

positive_rows = [row for row in rows if float(row["congestion_delay_total"]) > 0.0]
print(f"Rows with positive congestion delay: {len(positive_rows)} / {len(rows)}")
positive_rows[:3]


Rows with positive congestion delay: 6 / 12


[{'scenario_name': 'asymmetric_flow',
  'scenario_config': '/Users/adityadutta/Desktop/GitHub/GNN-Warehouse-Sim/configs/scenarios/asymmetric_flow.toml',
  'seed': '31',
  'policy': 'fifo',
  'execution_model': 'reserved_edges',
  'tasks_generated': '12',
  'tasks_completed': '12',
  'tasks_unassigned': '0',
  'average_waiting_time': '0.0',
  'average_turnaround_time': '38.49706017215655',
  'average_travel_distance_per_task': '13.0',
  'realized_travel_time_total': '161.9647220658787',
  'realized_travel_distance_total': '156.0',
  'congestion_delay_total': '5.9647220658787035',
  'average_congestion_delay_per_completed_task': '0.4970601721565586',
  'blocked_traversal_events_total': '2',
  'average_queue_length': '0.0',
  'throughput_per_hour': '57.36228208210263',
  'makespan': '753.1081127171308',
  'summary_path': '/Users/adityadutta/Desktop/GitHub/GNN-Warehouse-Sim/outputs/notebooks/e2e_cli/congestion_benchmark/asymmetric_flow/seed_31/fifo/summary.json'},
 {'scenario_name': 'asymm